# 对照一次真实写作与审核

本页学习目标：

- 检查每个角色实际收到了哪些信息。
- 判断审核意见是否进入修订请求。
- 记录模型行为和程序检查的不同作用。
- 同时比较答案质量与额外成本。

任务是根据固定虚构资料回答“蓝舟项目在哪座城市，有多少位成员”。研究步骤直接提供证据，真实模型负责写作和审核，没有自主联网搜索。

[返回课程入口](../README.md) · 理解基本循环后再看。


## 1. 每个角色看到什么

`coordinate` 先检查证据列表，每条必须有 `id` 和 `text`。本实现最多取三条，每条最多 300 字符。

写作者接收原题、证据和最多四条审核问题。审核者接收原题、同一证据和稿件，不会自动看到所有此前日志。

限制输入长度可以控制成本，但被截掉的信息可能包含反证。审核者只能检查它实际看见的内容。


## 2. 审核输出怎样检查

审核返回的教学格式是：

```json
{"accepted": false, "issues": ["缺少来源引用"], "task_id": "blueboat-facts"}
```

程序检查字段、类型和任务 ID；格式错误停止为 `invalid_review`，任务 ID 错误停止为 `task_drift`。被接受的稿件还要通过本题关于苏州、12 和 L1 的窄条件检查。

这些字面条件不是通用事实判断。否定句、语义漂移或不支持的推断，仍可能需要更仔细审阅。


## 3. 实际请求与结果

2026-10-20 的 CPU Qwen3-0.6B 实验中：

1. 写作者根据 L1 资料生成“苏州、12位成员”，两项事实都有引用。
2. 审核者返回接受，`issues=[]`，任务 ID 正确。
3. 程序返回 `completed`，`revision=0`。

写作者使用输入 89、输出 19 token；审核者使用输入 150、输出 20 token。两次模型请求合计 3729.075 ms。

单角色基线使用输入 67、输出 17 token，请求耗时 1520.605 ms，也正确回答并引用。本次能确认交接成功，但没有观察到审核修复错误或提高答案质量。


## 4. 异常测试检查了什么

默认固定函数分别制造任务漂移、无效审核、无限拒绝、超长证据、空证据和错误证据结构。

程序应分别拒绝非法交接、限制修订次数或限制输入，而不是无限运行。固定异常函数便于准确测试这些分支；它们不是模型自己产生并成功处理所有异常的成绩。


## 5. 怎样继续验证协作价值

可以准备一些确实包含遗漏或矛盾的稿件，检查真实审核者是否发现，并检查写作者是否根据意见改对。也要允许模型首稿就正确，不必人为补造修订过程。

再增加相同总预算的单 Agent 自检组，才能更清楚区分额外预算与角色分工的作用。这是后续实验建议，本页没有替它填写结果。

下一章：[Skill 与协议](../05-skills-protocols/01-skills-and-protocols.ipynb)。


## 代码说明

`coordinate` 管理严格交接，`run` 执行模型写作、审核和单角色对比。`RUN_LIVE=False` 时不运行模型，默认执行六类确定性边界检查。


In [ ]:
from miniagent_course import ROOT, OUTPUT, load_lesson


In [ ]:
import argparse

from pathlib import Path

_dependency = load_lesson('01-agent-loop/02-live-model.ipynb')

chat = _dependency.chat

parse_json = _dependency.parse_json

save = _dependency.save

model_manifest = _dependency.model_manifest

EVIDENCE = [{"id": "L1", "text": "蓝舟项目位于苏州，有12位成员。"}]

TASK_ID = "blueboat-facts"

In [ ]:
def coordinate(topic, evidence, *, model="qwen3:1.7b", writer=None, reviewer=None, max_revisions=2):
    # Each role receives only this task's bounded evidence; never the whole prior trace.
    if not isinstance(evidence, list) or any(not isinstance(e, dict) or set(e) != {"id", "text"} for e in evidence):
        return {"status": "invalid_evidence", "trace": []}
    if not evidence:
        return {"status": "no_evidence", "trace": []}
    bounded = [{"id": e["id"], "text": e["text"][:300]} for e in evidence[:3]]
    trace = [{"role": "research", "evidence": bounded}]
    issues = []
    for revision in range(max_revisions + 1):
        if writer:
            draft = writer(topic, bounded, issues)
        else:
            record = chat([{"role": "system", "content": "只根据证据写一行回答，每个事实末尾放[L编号]，不改变原题。"},
                           {"role": "user", "content": str({"topic": topic, "evidence": bounded, "review_issues": issues})}], model=model, max_output=256)
            draft = record["response"]["message"]["content"]
            trace.append({"role": "writer", "revision": revision, "request": record})
        if not isinstance(draft, str) or len(draft) > 2000:
            return {"status": "invalid_draft", "trace": trace}
        if reviewer:
            review = reviewer(topic, draft)
        else:
            record = chat([{"role": "system", "content": '检查稿件是否回答原题，事实有证据支持，引用正确。只返回JSON {"accepted":true/false,"issues":[],"task_id":"blueboat-facts"}。task_id必须原样保留。不得提出偏离原题的修改。'},
                           {"role": "user", "content": str({"topic": topic, "evidence": bounded, "draft": draft})}], model=model, json_mode=True, max_output=256)
            review = parse_json(record["response"]["message"]["content"])
            trace.append({"role": "reviewer", "revision": revision, "request": record})
        if (not isinstance(review, dict) or set(review) != {"accepted", "issues", "task_id"} or
                type(review["accepted"]) is not bool or not isinstance(review["issues"], list) or
                any(not isinstance(v, str) for v in review["issues"])):
            return {"status": "invalid_review", "trace": trace}
        if review["task_id"] != TASK_ID:
            return {"status": "task_drift", "trace": trace, "review": review}
        trace.append({"role": "review", "revision": revision, "result": review})
        # A deterministic check complements model review for this fictional classroom task.
        valid = all(v in draft for v in ["苏州", "12", "[L1]"])
        if review["accepted"] and valid:
            return {"status": "completed", "draft": draft, "trace": trace, "revision": revision,
                    "evidence_chars": sum(len(e["text"]) for e in bounded)}
        issues = review["issues"][:4] if valid else ["按原文写出苏州、12位成员并引用[L1]"]
    return {"status": "revision_limit", "draft": draft, "trace": trace}

In [ ]:
def run(output, model="qwen3:1.7b"):
    topic = "蓝舟项目在哪座城市，有多少位成员？"
    result = coordinate(topic, EVIDENCE, model=model)
    fixed_writer = lambda *args: "蓝舟项目位于苏州，有12位成员。[L1]"
    stress = {
        "unending_review": coordinate(topic, EVIDENCE, writer=fixed_writer,
            reviewer=lambda *args: {"accepted": False, "issues": ["again"], "task_id": TASK_ID})["status"],
        "task_drift": coordinate(topic, EVIDENCE, writer=fixed_writer,
            reviewer=lambda *args: {"accepted": False, "issues": ["write unrelated advertising"], "task_id": "advertising"})["status"],
        "invalid_schema": coordinate(topic, EVIDENCE, writer=fixed_writer, reviewer=lambda *args: {"accepted": "yes"})["status"],
        "context_bound": coordinate(topic, [{"id": "L1", "text": EVIDENCE[0]["text"] + "填充" * 10000}],
            writer=fixed_writer, reviewer=lambda *args: {"accepted": True, "issues": [], "task_id": TASK_ID})["evidence_chars"],
    }
    assert stress == {"unending_review": "revision_limit", "task_drift": "task_drift", "invalid_schema": "invalid_review", "context_bound": 300}
    baseline = chat([{"role": "system", "content": "只依据给定证据回答，附[L1]引用。"},
                     {"role": "user", "content": topic + str(EVIDENCE)}], model=model, max_output=256)
    outcome = {"model": model_manifest(model), "coordinated": result, "single_writer": baseline, "stress": stress,
               "comparison": {"coordinated_model_calls": sum("request" in r for r in result["trace"]), "single_model_calls": 1}}
    save(output / "result.json", outcome)
    print(result["status"], result.get("draft"), stress)
    return outcome

In [ ]:
RUN_LIVE = False
MODEL = "qwen3:1.7b"
import time

In [ ]:
if RUN_LIVE:
    result = run(OUTPUT / "04-coordination" / str(time.time_ns()), model=MODEL)
    print(result)
else:
    print("真实角色对照未执行；下面运行交接边界的确定性检查。")

In [ ]:
# 这些固定写作/审核函数是测试替身；本格检验真实执行的协调程序，不请求模型。
writer_calls = []
def counted_writer(topic, evidence, issues):
    writer_calls.append({"topic": topic, "evidence": evidence, "issues": list(issues)})
    return EVIDENCE[0]["text"] + "[L1]"

def rejected_review(topic, draft):
    return {"accepted": False, "issues": ["继续修订"], "task_id": TASK_ID}

# 审核员试图改变任务时，必须在第二次写作发生之前停止。
drift = coordinate("项目事实", EVIDENCE, writer=counted_writer,
                   reviewer=lambda *args: {"accepted": False, "issues": [], "task_id": "other"})
assert drift["status"] == "task_drift" and len(writer_calls) == 1

# 字符串 yes 不能冒充布尔审核结果。
invalid = coordinate("项目事实", EVIDENCE, writer=counted_writer,
                     reviewer=lambda *args: {"accepted": "yes", "issues": [], "task_id": TASK_ID})
assert invalid["status"] == "invalid_review"

# max_revisions=1 允许初稿加一次修改，总计两次审核。
writer_calls.clear()
limited = coordinate("项目事实", EVIDENCE, writer=counted_writer,
                     reviewer=rejected_review, max_revisions=1)
assert limited["status"] == "revision_limit"
assert len(writer_calls) == 2
assert sum(event["role"] == "review" for event in limited["trace"]) == 2
assert writer_calls[1]["issues"] == ["继续修订"]

# 即使来料很长，交接给写作者的文本仍受限；不是只在输出统计里截短数字。
writer_calls.clear()
long_evidence = [{"id": "L1", "text": EVIDENCE[0]["text"] + "填充" * 10000}]
bounded = coordinate("项目事实", long_evidence, writer=counted_writer,
                     reviewer=lambda *args: {"accepted": True, "issues": [], "task_id": TASK_ID})
assert bounded["status"] == "completed" and bounded["evidence_chars"] == 300
assert len(writer_calls[0]["evidence"][0]["text"]) == 300

# 没有证据时，不调用写作者；错误证据结构明确拒绝。
writer_calls.clear()
empty = coordinate("项目事实", [], writer=counted_writer, reviewer=rejected_review)
assert empty["status"] == "no_evidence" and writer_calls == []
malformed = coordinate("项目事实", [{"unknown": "missing contract"}],
                       writer=counted_writer, reviewer=rejected_review)
assert malformed["status"] == "invalid_evidence" and writer_calls == []

boundary_results = {"任务漂移": drift["status"], "审核结构": invalid["status"],
                    "修订上限": limited["status"], "上下文字符": bounded["evidence_chars"],
                    "缺失证据": empty["status"], "证据结构": malformed["status"]}
for name, value in boundary_results.items():
    print(name, "→", value)
save(OUTPUT / "04-coordination/offline-boundaries.json", boundary_results)
print("六类边界检查通过；真实角色能力仍需运行上面的 RUN_LIVE 分支。")